# TODO Agent (LangGraph v1 + InMemoryStore)

In [4]:
# !pip install langchain langgraph langchain-openai python-dotenv

import os
import uuid

from typing import Protocol

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.config import get_store
from langgraph.store.memory import InMemoryStore

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if OPENAI_API_KEY:
    api_key = OPENAI_API_KEY
    base_url = None
elif OPENROUTER_API_KEY:
    api_key = OPENROUTER_API_KEY
    base_url = "https://openrouter.ai/api/v1"
else:
    raise RuntimeError(
        "OPENAI_API_KEY or OPENROUTER_API_KEY is required to run this notebook"
    )

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

llm_kwargs = {"model": MODEL, "temperature": 0, "api_key": api_key}
if base_url:
    llm_kwargs["base_url"] = base_url

llm = ChatOpenAI(**llm_kwargs)

In [5]:
USER_ID = os.getenv("TODO_USER_ID", "default")
NAMESPACE = ("todo", USER_ID)

STORE = InMemoryStore()


class StoreItem(Protocol):
    key: str
    value: dict[str, str]


class TodoStore(Protocol):
    def put(
        self, namespace: tuple[str, str], key: str, value: dict[str, str]
    ) -> None: ...
    def get(self, namespace: tuple[str, str], key: str) -> StoreItem | None: ...
    def search(
        self, namespace: tuple[str, str], **kwargs: object
    ) -> list[StoreItem]: ...
    def delete(self, namespace: tuple[str, str], key: str) -> None: ...


class TaskData(BaseModel):
    text: str = Field(min_length=1, description="Task description.")


class TaskView(BaseModel):
    task_id: str = Field(min_length=1, description="Full task UUID.")
    text: str = Field(min_length=1, description="Task description.")


class AddTaskInput(BaseModel):
    text: str = Field(min_length=1, description="Task description.")


class DeleteTaskInput(BaseModel):
    task_id: str = Field(min_length=1, description="Task id to delete.")


def _search_all(
    store: TodoStore, namespace: tuple[str, str], limit: int = 100
) -> list[StoreItem]:
    try:
        return store.search(namespace, limit=limit)
    except TypeError:
        return store.search(namespace, query="", limit=limit)


def _task_from_item(item: StoreItem) -> TaskView:
    data = TaskData.model_validate(item.value)
    return TaskView(task_id=item.key, text=data.text)


def _format_task_line(index: int, task: TaskView) -> str:
    return f"{index}. {task.text} [ID: {task.task_id}]"


@tool("add_task", args_schema=AddTaskInput)
def add_task(text: str) -> str:
    """Add a new task to the store."""
    store = get_store()
    task = TaskData(text=text.strip())
    task_id = str(uuid.uuid4())
    store.put(NAMESPACE, task_id, task.model_dump())
    return f'Завдання "{task.text}" було успішно додано. ID: {task_id}'


@tool("list_tasks")
def list_tasks() -> str:
    """List all tasks from the store."""
    store = get_store()
    items = _search_all(store, NAMESPACE, limit=100)
    if not items:
        return "Список завдань порожній."
    lines = ["Ось всі ваші завдання:"]
    for idx, item in enumerate(items, 1):
        task = _task_from_item(item)
        lines.append(_format_task_line(idx, task))
    return "\n".join(lines)


@tool("delete_task", args_schema=DeleteTaskInput)
def delete_task(task_id: str) -> str:
    """Delete a task by id from the store."""
    store = get_store()
    item = store.get(NAMESPACE, task_id)
    if item is None:
        return f"Завдання з ID {task_id} не знайдено."

    task = _task_from_item(item)
    store.delete(NAMESPACE, task_id)
    return f'Завдання "{task.text}" було видалено.'

In [6]:
SYSTEM_PROMPT = """
Ти TODO-агент. Відповідай українською, коротко і по суті.
Використовуй інструменти для додавання, перегляду і видалення задач.
Якщо користувач просить видалити задачу за описом, спершу виклич list_tasks,
знайди відповідний повний ID, а потім виклич delete_task з цим ID.
""".strip()

TOOLS = [add_task, list_tasks, delete_task]

agent = create_agent(
    llm,
    tools=TOOLS,
    store=STORE,
    system_prompt=SYSTEM_PROMPT,
)

THREAD_ID = "todo-thread-1"


def run_agent(text: str) -> str:
    result = agent.invoke(
        {"messages": [("user", text)]},
        config={"configurable": {"thread_id": THREAD_ID, "user_id": USER_ID}},
    )
    last = result["messages"][-1]
    return last.content if hasattr(last, "content") else str(last)


tests = [
    "Додай: купити хліб",
    "Додай: подзвонити лікарю",
    "Покажи всі завдання",
    "Видали завдання про хліб",
    "Що залишилось?",
]

for i, test in enumerate(tests, 1):
    print(f"\n{'='*50}")
    print(f"TEST {i}: {test}")
    print("=" * 50)
    print(run_agent(test))


TEST 1: Додай: купити хліб
Завдання "купити хліб" успішно додано.

TEST 2: Додай: подзвонити лікарю
Завдання "подзвонити лікарю" успішно додано.

TEST 3: Покажи всі завдання
Ось всі ваші завдання:
1. купити хліб [ID: e51b4f8f-a25f-429b-ad51-943ce1018576]
2. подзвонити лікарю [ID: e60fa9fc-1bd6-4aa3-8b36-db175a6e59ba]

TEST 4: Видали завдання про хліб
Завдання "купити хліб" успішно видалено.

TEST 5: Що залишилось?
У вас залишилось одне завдання: подзвонити лікарю.
